### Start vLLM server (run in a separate terminal before executing inference cells)

```bash
vllm serve /gpfs/projects/bsc100/models/DeepSeek-R1-Distill-Qwen-32B \
    --tensor-parallel-size 4 \
    --max-model-len 32000 \
    --enable-chunked-prefill \
    --max-num-batched-tokens 32000 \
    --reasoning-parser deepseek_r1 \
    --gpu-memory-utilization 0.8 \
    --port 8000
```

Wait until you see `INFO: Application startup complete.` before running the cells below.

In [1]:
from pathlib import Path
import sys

def find_project_root(start_path=None, marker=".git"):
    path = Path(start_path or Path.cwd()).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

ROOT_DIR = find_project_root()
print(ROOT_DIR)


if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

/gpfs/home/bsc/bsc739841/catalanGraph


In [2]:
import os
# Force the system to bypass proxies for local connections
os.environ["NO_PROXY"] = "127.0.0.1,localhost"

from openai import OpenAI
import httpx

MODEL_PATH = "/gpfs/projects/bsc100/models/DeepSeek-R1-Distill-Qwen-32B"
INFERENCE_PARAMS = {
    "temperature": 0.4,
    "top_p": 0.90,
    "max_tokens": 4096,
    "extra_body": {"repetition_penalty": 1.1}
}

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="none", http_client=httpx.Client(proxy=None))

def inference_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL_PATH,
        messages=[{"role": "user", "content": prompt}],
        **INFERENCE_PARAMS
    )
    think = response.choices[0].message.reasoning_content
    answer = response.choices[0].message.content
    return think, answer

In [3]:
import json

json_path = "./data/catalan_subgraph.json"

print(f"Loading graph from {json_path}...")
with open(json_path, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

documentSections= []
for node in graph_data.get("nodes", []):
    if "DocumentSection" in node["labels"]:
        documentSections.append(node)


print(f"Total nodes loaded: {len(graph_data.get('nodes', []))}")
print(f"Total section loaded: {len(documentSections)}")


Loading graph from ./data/catalan_subgraph.json...
Total nodes loaded: 5000
Total section loaded: 45


In [4]:
# Select a node by index to infer something on
selected_index = 0

if selected_index < len(documentSections):
    node = documentSections[selected_index]
    props = node.get("properties", {})

# Extract text and details
    text_to_analyze = props.get("textEs") or props.get("textCa")
    lang = "Spanish (textEs)" if "textEs" in props else "Catalan (textCa)"
    title = props.get("titleEs") or props.get("titleCa") or "No Title"
    node_id = node.get("id")
    labels = node.get("labels", [])

    # Custom inference query / question
    inference_query = "Provide a list of all legal documents cited by the Source Legal Text as a dictionary (report document cited, the type of the citation (general, modification, abrogation) and where in the text)."
    
    prompt = f"""You are analyzing a the text from a section of a legal document from the catalan gazette.
Title: {title}
Language of source text: {lang}

Source Legal Text:
\"\"\"
{text_to_analyze}
\"\"\"

Task:
{inference_query}
"""
    
    print(f"--- Prompting model on Node ID {node_id} ({title}) ---")
    print(f"Query: {inference_query}\n")
    
    # Run the inference call
    think, answer = inference_llm(prompt)
    
    print("=== Thinking Process ===")
    print(think)
    print("\n=== Model Answer ===")
    print(answer)
else:
    print(f"Error: Selected index {selected_index} is out of bounds (0 to {len(documentSections)-1}).")



--- Prompting model on Node ID 1479 (Preamble) ---
Query: Provide a list of all legal documents cited by the Source Legal Text as a dictionary (report document cited, the type of the citation (general, modification, abrogation) and where in the text).

=== Thinking Process ===
Okay, so I need to figure out how to respond to this query about analyzing a legal document from the Catalan Gazette. The user provided a preamble in Spanish and wants me to extract the cited legal documents along with their types and locations in the text.

First, I'll read through the source text carefully. It mentions "el artículo 12.1 k)" and "el artículo 20" from "Ley 13/2008, de 5 de noviembre, de la presidencia de la Generalidad y del Gobierno." So that's one document cited twice.

Next, I have to determine the type of each citation. Since both articles are referenced without any indication of being modified or repealed, it seems they're just being cited for their content. So both would be 'general' citati

In [ ]:
import re
import json

def extract_graph_citations(node_id, graph_data):
    """1) Extract list of citations from graph for node_id (target id, labels, title) and count."""
    nodes_by_id = {n["id"]: n for n in graph_data.get("nodes", [])}
    citations = []
    citation_types = {"CITES", "AFFECTS", "ABROGATES", "MODIFIES", "CONSOLIDATES"}
    
    for rel in graph_data.get("relationships", []):
        if rel.get("source") == node_id and rel.get("type") in citation_types:
            target_id = rel["target"]
            target_node = nodes_by_id.get(target_id, {})
            
            # Extract title (fallback to relationship citedDocument if target title is missing)
            title = (target_node.get("properties", {}).get("titleEs") or 
                     target_node.get("properties", {}).get("titleCa") or 
                     target_node.get("properties", {}).get("title") or 
                     rel.get("properties", {}).get("citedDocument", "No Title"))
            labels = target_node.get("labels", [])
            
            citations.append({
                "target_id": target_id,
                "labels": labels,
                "title": title
            })
            
    return citations, len(citations)

def extract_llm_citations(llm_output):
    """2) Reads the JSON of the output of LLM, gets the document titles and count."""
    if isinstance(llm_output, str):
        try:
            # Strip code block backticks if present
            match = re.search(r"```(?:json)?\s*(.*?)\s*```", llm_output, re.DOTALL | re.IGNORECASE)
            content = match.group(1) if match else llm_output
            data = json.loads(content.strip())
        except Exception:
            return [], 0
    else:
        data = llm_output
        
    titles = []
    if isinstance(data, list):
        items = data
    elif isinstance(data, dict):
        # Check if list of citations is nested under some key
        items = next((v for v in data.values() if isinstance(v, list)), [data])
    else:
        items = []
        
    for item in items:
        if isinstance(item, dict):
            # Find the most likely key for the document title
            title = (item.get("document_title") or 
                     item.get("Cited Document") or 
                     item.get("document") or 
                     item.get("document_cited") or 
                     item.get("documentCited") or 
                     item.get("title"))
            if not title:
                # Fallback to the first string value longer than 5 chars
                title = next((v for v in item.values() if isinstance(v, str) and len(v) > 5), None)
            if title:
                titles.append(title)
        elif isinstance(item, str):
            titles.append(item)
            
    return titles, len(titles)

def compare_and_print_citations(graph_citations, llm_titles):
    """3) Compares numbers and aligns/prints all citations."""
    def get_numbers(text):
        return set(re.findall(r"\b\d+/\d{4}\b", text))
        
    print("=" * 60)
    print(f"CITATION COMPARISON REPORT")
    print(f"Graph Citations Count: {len(graph_citations)} | LLM Citations Count: {len(llm_titles)}")
    print("-" * 60)
    
    matched_graph_indices = set()
    matched_llm_indices = set()
    alignments = []
    
    # Try to align based on numbers (e.g. 68/2024) or substring containment
    for g_idx, g_cit in enumerate(graph_citations):
        g_title = g_cit["title"]
        g_nums = get_numbers(g_title)
        
        for l_idx, l_title in enumerate(llm_titles):
            if l_idx in matched_llm_indices:
                continue
            l_nums = get_numbers(l_title)
            
            # Check match by numbers or substring containment
            if ((g_nums & l_nums) or 
                (l_title.lower().strip() in g_title.lower().strip()) or 
                (g_title.lower().strip() in l_title.lower().strip())):
                alignments.append((g_cit, l_title))
                matched_graph_indices.add(g_idx)
                matched_llm_indices.add(l_idx)
                break
                
    # Print Aligned Matches
    print("ALIGNED CITATIONS (MATCHES):")
    for g_cit, l_title in alignments:
        print(f"  [MATCH]")
        print(f"    Graph: ID={g_cit['target_id']}, Labels={g_cit['labels']}, Title={g_cit['title']}")
        print(f"    LLM:   Title={l_title}")
        
    # Print Missing in LLM (LESS)
    missing = [g for idx, g in enumerate(graph_citations) if idx not in matched_graph_indices]
    if missing:
        print(f"\nMISSING IN LLM (In Graph, but not LLM):")
        for g_cit in missing:
            print(f"  [-] ID={g_cit['target_id']}, Labels={g_cit['labels']}, Title={g_cit['title']}")
            
    # Print Extra in LLM (MORE)
    extra = [l for idx, l in enumerate(llm_titles) if idx not in matched_llm_indices]
    if extra:
        print(f"\nEXTRA IN LLM (In LLM, but not Graph):")
        for l_title in extra:
            print(f"  [+] Title={l_title}")
            
    print("=" * 60)
